# 01. Preguntas, evidencias y qrels

Las métricas de recuperación necesitan una referencia de relevancia independiente de la estrategia evaluada. Este cuaderno relaciona las formulaciones con los fragmentos juzgados y comprueba la cobertura de esa referencia. No añade ni modifica decisiones.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

In [ ]:
golden = pd.DataFrame(read_jsonl(ROOT / 'data/golden/variants/generation_golden_v3.jsonl'))
qrels = pd.DataFrame(read_jsonl(ROOT / 'outputs/local_runs/explicitness_v3/qrels_final_v3.jsonl'))
chunks = pd.DataFrame(read_jsonl(ROOT / 'data/processed/chunks_v1.jsonl'))
assert len(golden) == 52 and golden['base_qid'].nunique() == 26
assert len(qrels) == qrels['unit_id'].nunique() == 578
assert not qrels.duplicated(['base_qid', 'chunk_id']).any()
assert set(qrels['base_qid']) == set(golden['base_qid'])
assert set(qrels['chunk_id']).issubset(set(chunks['chunk_id']))

## Formulación y conducta esperada

La conducta esperada distingue una respuesta ordinaria, la corrección de una premisa falsa y la abstención por falta de evidencia. Una pregunta con premisa falsa puede necesitar fragmentos que permitan refutarla; no se trata igual que una pregunta sin evidencia disponible.

In [ ]:
assert golden.groupby('base_qid')['expected_behavior'].nunique().eq(1).all()
bases = golden.drop_duplicates('base_qid')
display(bases['expected_behavior'].value_counts().rename_axis('conducta').to_frame('preguntas_base'))
assert bases['expected_behavior'].value_counts().to_dict() == {'answer': 17, 'grounded_correction': 5, 'abstain': 4}
display(pd.crosstab(golden['explicitness'], golden['expected_behavior']))

Las dos formulaciones comparten la pregunta base, pero cambian su explicitud. La tabla permite revisar el texto exacto antes de interpretar una diferencia de rendimiento.

In [ ]:
pairs = golden.pivot(index='base_qid', columns='explicitness', values='question')
assert pairs.notna().all().all()
display(pairs)

## Grados y procedencia de la relevancia

El grado 2 representa evidencia suficiente; el grado 1, contexto relacionado; el grado 0, material no relevante. Recall, Precision y MRR utilizan el criterio estricto de grado 2. nDCG conserva los grados 0/1/2 y calcula su ideal con toda la referencia conocida de la pregunta.

Los qrels finales se identifican por pregunta base y fragmento. El evaluador resuelve esa correspondencia para las dos formulaciones. La procedencia de cada decisión permanece en grade_source: un consenso automático no se presenta como una revisión humana.

In [ ]:
display(pd.crosstab(qrels['grade_source'], qrels['grade'], margins=True))
display(qrels.groupby('pass_tag').agg(unidades=('unit_id', 'size'),
                                      preguntas_base=('base_qid', 'nunique'),
                                      fragmentos=('chunk_id', 'nunique')))

La cobertura de grado 2 varía entre preguntas. Los recuentos siguientes describen la referencia disponible; no afirman que se hayan juzgado todos los fragmentos del corpus.

In [ ]:
coverage = pd.crosstab(qrels['base_qid'], qrels['grade']).reindex(columns=[0, 1, 2], fill_value=0)
coverage.columns = ['grado_0', 'grado_1', 'grado_2']
coverage = coverage.join(bases.set_index('base_qid')[['expected_behavior']])
display(coverage)

## Una evidencia completa

El ejemplo se selecciona por identificador para que pueda contrastarse con el corpus. Su grado ya está decidido en los artefactos incluidos; leerlo aquí no constituye una nueva adjudicación.

In [ ]:
unit = qrels.loc[(qrels['base_qid'] == 'q-v1-001') & (qrels['grade'] == 2)].iloc[0]
text = chunks.set_index('chunk_id').loc[unit['chunk_id'], 'retrieval_text']
display(golden.loc[golden['base_qid'] == unit['base_qid'],
                   ['qid', 'question', 'reference', 'expected_behavior']])
display(unit[['base_qid', 'chunk_id', 'grade', 'grade_source', 'pass_tag']].to_frame('valor'))
print(text)

Los qrels permanecen congelados durante el reanálisis. El cuaderno siguiente utiliza esta misma referencia para todas las estrategias y comprueba los candidatos que conserva el reranking.